# End-to-End Fraud Detection Pipeline & Model Benchmarking
This notebook implements exploratory data analysis, spatial-temporal feature engineering, imbalanced learning strategies, and multi-model benchmarking (Logistic Regression, XGBoost, LightGBM) with MLflow experiment tracking using dedicated training (`fraudTrain.csv`) and out-of-time testing (`fraudTest.csv`) datasets.

In [ ]:
import os
import json
from contextlib import nullcontext
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier
import lightgbm as lgb

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False

## 1. Data Ingestion (`fraudTrain.csv` & `fraudTest.csv`)

In [ ]:
TRAIN_PATH = '/kaggle/input/fraud-detection/fraudTrain.csv'
TEST_PATH = '/kaggle/input/fraud-detection/fraudTest.csv'

if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = '../data/raw/fraudTrain.csv'
if not os.path.exists(TEST_PATH):
    TEST_PATH = '../data/raw/fraudTest.csv'

# Fallback to sample_transactions.csv if full datasets are not yet placed in data/raw/
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = '../data/raw/sample_transactions.csv'
if not os.path.exists(TEST_PATH):
    TEST_PATH = '../data/raw/sample_transactions.csv'

# Set SAMPLE_FRAC = 1.0 to run on 100% of the data, or 0.20 for faster iteration
SAMPLE_FRAC = 0.20
RANDOM_STATE = 42

def load_and_sample_dataset(csv_path: str, sample_frac: float = 0.20) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if 0.0 < sample_frac < 1.0 and len(df) > 20000:
        df = (
            df.groupby('is_fraud', group_keys=False)
            .apply(lambda x: x.sample(frac=sample_frac, random_state=RANDOM_STATE))
            .reset_index(drop=True)
        )
    return df

train_raw = load_and_sample_dataset(TRAIN_PATH, SAMPLE_FRAC)
test_raw = load_and_sample_dataset(TEST_PATH, SAMPLE_FRAC)

print(f"Train dataset shape: {train_raw.shape} | Fraud cases: {train_raw['is_fraud'].sum():,}")
print(f"Test dataset shape : {test_raw.shape} | Fraud cases: {test_raw['is_fraud'].sum():,}")
train_raw.head()

## 2. Spatial-Temporal Feature Engineering

In [ ]:
FEATURE_ORDER = [
    'merchant', 'category', 'amt', 'gender', 'city_pop', 'job',
    'age', 'age_group', 'day', 'month', 'year', 'hour', 'minute',
    'distance_km', 'distance_group'
]

CATEGORICAL_COLS = ['merchant', 'category', 'gender', 'job', 'age_group', 'distance_group']
NUMERICAL_COLS = ['amt', 'city_pop', 'age', 'day', 'month', 'year', 'hour', 'minute', 'distance_km']

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    return R * 2.0 * np.arcsin(np.sqrt(a))

def age_group(age):
    if age < 18: return 'Teen'
    elif age < 25: return 'Young Adult'
    elif age < 35: return 'Adult'
    elif age < 50: return 'Middle-aged'
    elif age < 65: return 'Senior'
    return 'Elderly'

def distance_category(dist):
    if dist < 0.5: return 'Very Near'
    elif dist < 3.0: return 'Near'
    elif dist < 15.0: return 'Moderate'
    elif dist < 50.0: return 'Far'
    return 'Very Far'

def build_features_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data['dob'] = pd.to_datetime(data['dob'])
    data['trans_date_trans_time'] = pd.to_datetime(data['trans_date_trans_time'])

    data['age'] = (data['trans_date_trans_time'].dt.year - data['dob'].dt.year).astype(int)
    data['age_group'] = data['age'].apply(age_group)

    data['day'] = data['trans_date_trans_time'].dt.day.astype(int)
    data['month'] = data['trans_date_trans_time'].dt.month.astype(int)
    data['year'] = data['trans_date_trans_time'].dt.year.astype(int)
    data['hour'] = data['trans_date_trans_time'].dt.hour.astype(int)
    data['minute'] = data['trans_date_trans_time'].dt.minute.astype(int)

    data['distance_km'] = haversine(data['lat'], data['long'], data['merch_lat'], data['merch_long'])
    data['distance_group'] = data['distance_km'].apply(distance_category)

    cols_to_drop = [
        'Unnamed: 0', 'first', 'last', 'street', 'city', 'state', 'zip',
        'dob', 'cc_num', 'trans_num', 'unix_time',
        'lat', 'long', 'merch_lat', 'merch_long', 'trans_date_trans_time'
    ]
    data.drop(columns=[c for c in cols_to_drop if c in data.columns], inplace=True)
    return data

train_processed = build_features_dataframe(train_raw)
test_processed = build_features_dataframe(test_raw)

print('Processed feature columns:', list(train_processed.columns))

## 3. Feature Encoding & Numerical Scaling (Fitted on Train, Transformed on Test)

In [ ]:
X_train = train_processed[FEATURE_ORDER].copy()
y_train = train_processed['is_fraud'].astype(int)

X_test = test_processed[FEATURE_ORDER].copy()
y_test = test_processed['is_fraud'].astype(int)

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train[CATEGORICAL_COLS] = encoder.fit_transform(X_train[CATEGORICAL_COLS])
X_test[CATEGORICAL_COLS] = encoder.transform(X_test[CATEGORICAL_COLS])

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[NUMERICAL_COLS] = scaler.fit_transform(X_train[NUMERICAL_COLS])
X_test_scaled[NUMERICAL_COLS] = scaler.transform(X_test[NUMERICAL_COLS])

print(f"X_train prepared: {X_train.shape} | X_test prepared: {X_test.shape}")

## 4. Multi-Model Training, Out-of-Time Evaluation & MLflow Experiment Tracking

In [ ]:
neg_class = int((y_train == 0).sum())
pos_class = max(int((y_train == 1).sum()), 1)
class_weight_ratio = neg_class / pos_class

models = {
    'Baseline_LogisticRegression': LogisticRegression(
        solver='lbfgs', penalty='l2', class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        n_estimators=250, max_depth=6, learning_rate=0.05,
        scale_pos_weight=class_weight_ratio, subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'LightGBM': lgb.LGBMClassifier(
        objective='binary', class_weight='balanced',
        n_estimators=250, learning_rate=0.05, max_depth=6, num_leaves=31,
        random_state=RANDOM_STATE, n_jobs=-1, verbose=-1
    )
}

if MLFLOW_AVAILABLE:
    mlflow.set_experiment('Fraud_Detection_Production_Benchmark')

results = []
best_f1 = -1.0
best_model = None
best_model_name = ''
best_metrics = {}

for name, model in models.items():
    run_ctx = mlflow.start_run(run_name=name) if MLFLOW_AVAILABLE else nullcontext()
    with run_ctx:
        X_tr = X_train_scaled if name == 'Baseline_LogisticRegression' else X_train
        X_te = X_test_scaled if name == 'Baseline_LogisticRegression' else X_test

        model.fit(X_tr, y_train)
        y_pred = model.predict(X_te)
        y_prob = model.predict_proba(X_te)[:, 1]

        prec = float(precision_score(y_test, y_pred, zero_division=0))
        rec = float(recall_score(y_test, y_pred, zero_division=0))
        f1 = float(f1_score(y_test, y_pred, zero_division=0))
        roc_auc = float(roc_auc_score(y_test, y_prob))
        acc = float(accuracy_score(y_test, y_pred))
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        if MLFLOW_AVAILABLE:
            mlflow.log_params(model.get_params())
            mlflow.log_metrics({'precision': prec, 'recall': rec, 'f1_score': f1, 'roc_auc': roc_auc, 'accuracy': acc})
            mlflow.sklearn.log_model(model, artifact_path='model')

        results.append({
            'Model': name, 'Precision': round(prec, 4), 'Recall': round(rec, 4),
            'F1-Score': round(f1, 4), 'ROC-AUC': round(roc_auc, 4), 'Accuracy': round(acc, 4),
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn)
        })
        if f1 > best_f1:
            best_f1 = f1
            best_model = model
            best_model_name = name
            best_metrics = {'precision': round(prec, 4), 'recall': round(rec, 4), 'f1_score': round(f1, 4), 'roc_auc': round(roc_auc, 4), 'accuracy': round(acc, 4)}

comparison_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
print(comparison_df.to_string(index=False))

# Export serialized artifacts for FastAPI deployment
os.makedirs('../models', exist_ok=True)
joblib.dump(encoder, '../models/ordinal_encoder.pkl')
joblib.dump(scaler, '../models/standard_scaler.pkl')
joblib.dump(best_model, '../models/best_fraud_model.pkl')

metadata = {
    'best_model_name': best_model_name,
    'requires_scaling': best_model_name == 'Baseline_LogisticRegression',
    'metrics': best_metrics,
    'feature_order': FEATURE_ORDER,
    'categorical_columns': CATEGORICAL_COLS,
    'numerical_columns': NUMERICAL_COLS,
    'benchmark_table': results
}
with open('../models/model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved champion model ({best_model_name}) and preprocessors to ../models/")